# Analysis of Cursor Movement basic reaching

In [36]:
import pandas as pd
from pathlib import Path
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output

# Creation of Data Frame

In [51]:
# -----------------------------------------------
# 1. Alle CSVs laden und Metadaten anfügen
# -----------------------------------------------
basis_ordner = Path("data/Basic_reaching")
alle_dfs = []

blob_mapping = {0: 0.15, 1: 0.3, 2: 0.45, 3: 0.6}

for datei_pfad in basis_ordner.rglob("*.csv"):
    df_temp = pd.read_csv(datei_pfad)
    df_temp['Participant'] = datei_pfad.parent.name
    
    if 'block_' in datei_pfad.name:
        try:
            block_str = datei_pfad.name.split('block_')[1].split('_')[0]
            block = int(block_str)
        except (IndexError, ValueError):
            block = -1
            print(f" Unerwartetes Format, als Training eingestuft: {datei_pfad.name}")
    else:
        block = -1
        print(f" Kein 'block_' im Namen, als Training eingestuft: {datei_pfad.name}")
    
    df_temp['block'] = block
    df_temp['blob_width'] = blob_mapping.get(block, None)
    df_temp['is_training'] = (block == -1)
    
    alle_dfs.append(df_temp)

df = pd.concat(alle_dfs, ignore_index=True)

# -----------------------------------------------
# 2. Training entfernen & Gaze-Ausreißer maskieren
# -----------------------------------------------
df_cc = df[df['is_training'] == False].copy()

mask_invalid_x = (df_cc['gaze_x'] < -3000) | (df_cc['gaze_x'] > 3000)
mask_invalid_y = (df_cc['gaze_y'] < -3000) | (df_cc['gaze_y'] > 3000)
df_cc.loc[mask_invalid_x, 'gaze_x'] = np.nan
df_cc.loc[mask_invalid_y, 'gaze_y'] = np.nan

print(" df_cc erstellt:", df_cc.shape)

 Kein 'block_' im Namen, als Training eingestuft: cg66_1_2026-06-29_17h43.24.042.csv
 Kein 'block_' im Namen, als Training eingestuft: ub51_1_2026-06-29_18h35.51.281.csv
 Kein 'block_' im Namen, als Training eingestuft: ub51_1_2026-06-29_18h36.36.958.csv


C:\Users\march\AppData\Local\Temp\ipykernel_8764\4186068798.py:30: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(alle_dfs, ignore_index=True)


 df_cc erstellt: (822811, 31)


## Daten glätten um spünge der cursor daten zu verhindern 

In [52]:
def compute_radial_velocity(df, smooth_window=10):
    """
    Berechnet radiale Geschwindigkeit aus den Rohdaten.
    Verwendet State 2-3-4, robuste Glättung (Median + gleitender Mittelwert),
    und den korrekten Startpunkt (erster Frame in State 2).
    """
    # Nur State 2, 3, 4
    df = df[df['state_marker'].isin([2, 3, 4])].copy()
    df = df.sort_values(['Participant', 'block', 'trial', 'time'])

    # 1. Median-Filter (3er Fenster)
    df['cursor_x_med'] = df.groupby(['Participant', 'block', 'trial'])['cursor_x_pix'].transform(
        lambda x: x.rolling(window=3, center=True, min_periods=1).median()
    )
    df['cursor_y_med'] = df.groupby(['Participant', 'block', 'trial'])['cursor_y_pix'].transform(
        lambda x: x.rolling(window=3, center=True, min_periods=1).median()
    )

    # 2. Gleitender Mittelwert auf median-gefilterten Daten
    df['cursor_x_smooth'] = (
        df.groupby(['Participant', 'block', 'trial'])['cursor_x_med']
        .rolling(window=smooth_window, center=True, min_periods=1)
        .mean()
        .reset_index(level=[0, 1, 2], drop=True)
    )
    df['cursor_y_smooth'] = (
        df.groupby(['Participant', 'block', 'trial'])['cursor_y_med']
        .rolling(window=smooth_window, center=True, min_periods=1)
        .mean()
        .reset_index(level=[0, 1, 2], drop=True)
    )

    # Zeitliche Differenzen
    df['dt'] = df.groupby(['Participant', 'block', 'trial'])['time'].diff()
    df['dx'] = df.groupby(['Participant', 'block', 'trial'])['cursor_x_smooth'].diff()
    df['dy'] = df.groupby(['Participant', 'block', 'trial'])['cursor_y_smooth'].diff()
    df['vx'] = df['dx'] / df['dt']
    df['vy'] = df['dy'] / df['dt']

    # Startposition (State 2)
    first_state2 = df[df['state_marker'] == 2].groupby(['Participant', 'block', 'trial']).first()
    start_x = first_state2['cursor_x_smooth']
    start_y = first_state2['cursor_y_smooth']
    targets = df.groupby(['Participant', 'block', 'trial']).first()[['target_x', 'target_y']]
    dx_target = targets['target_x'] - start_x
    dy_target = targets['target_y'] - start_y
    dist = np.sqrt(dx_target**2 + dy_target**2)
    ux = dx_target / dist
    uy = dy_target / dist

    df = df.merge(pd.DataFrame({'ux': ux, 'uy': uy, 'dist_to_target': dist}, index=ux.index),
                  on=['Participant', 'block', 'trial'], how='left')

    # Geschwindigkeitskomponenten begrenzen
    df.loc[df['vx'].abs() > 3000, ['vx', 'vy']] = np.nan
    df.loc[df['vy'].abs() > 3000, ['vx', 'vy']] = np.nan

    # Radiale Geschwindigkeit
    df['v_radial'] = df['vx'] * df['ux'] + df['vy'] * df['uy']
    df.loc[df['v_radial'].abs() > 3250, 'v_radial'] = np.nan

    # Relative Zeit in State 2-3-4
    df['time_rel'] = df.groupby(['Participant', 'block', 'trial'])['time'].transform(lambda x: x - x.min())

    return df

# Anwenden
df_cc = compute_radial_velocity(df_cc, smooth_window=10)

print(" Velocity berechnet. Spalten:", [c for c in df_cc.columns if 'v_rad' in c])

 Velocity berechnet. Spalten: ['v_radial']


# computing cursor metrics

In [56]:
def compute_cursor_metrics(trial_data, threshold=10, cutoff_s=0.05):
    """
    Cursor-Metriken für einen Trial (State 2–4) mit Cutoff am Ende.
    Präzision = mittlerer Abstand der Cursor-Position im State 4 zum Target.
    """
    df = trial_data[trial_data['state_marker'].isin([2, 3, 4])].copy()
    if df.empty:
        return None

    time_rel = df['time'] - trial_data['time'].iloc[0]
    max_t = time_rel.max()
    keep = time_rel <= (max_t - cutoff_s)
    df = df[keep].copy()
    time_rel = time_rel[keep]

    if df.empty:
        return None

    t_end = time_rel.max()
    latency = peak = duration = quick_phase = slow_phase = pos_error = precision = np.nan

    mask = df['v_radial'].fillna(0) > threshold
    if mask.any():
        latency = time_rel[mask].iloc[0]
        t0 = latency
        t_peak = time_rel[df['v_radial'].idxmax()]
        duration = t_end - t0
        quick_phase = t_peak - t0
        slow_phase = t_end - t_peak

    peak = df['v_radial'].max()

    target_x = df['target_x'].iloc[0]
    target_y = df['target_y'].iloc[0]

    # Position error = letzte Cursor-Position zum Target
    last = df.iloc[-1]
    pos_error = np.sqrt((last['cursor_x_smooth'] - target_x)**2 +
                        (last['cursor_y_smooth'] - target_y)**2)

    # Präzision = mittlerer Abstand aller Cursor-Punkte im State 4 zum Target
    df_state4 = df[df['state_marker'] == 4]
    if len(df_state4) > 0:
        dists = np.sqrt((df_state4['cursor_x_smooth'] - target_x)**2 +
                        (df_state4['cursor_y_smooth'] - target_y)**2)
        precision = dists.mean()

    return {
        'latency_cursor': latency,
        'peak_velocity_cursor': peak,
        'duration_cursor': duration,
        'quick_phase_cursor': quick_phase,
        'slow_phase_cursor': slow_phase,
        'position_error_cursor': pos_error,
        'precision_cursor': precision
    }

# compute gaze metrics

In [57]:
def compute_gaze_metrics_for_trial(trial_data, threshold=10, cutoff_s=0.05):
    """
    Gaze-Metriken für einen Trial (State 2–4) mit Cutoff am Ende.
    Präzision = mittlerer Abstand der geglätteten Gaze im State 4 zum Target.
    """
    df = trial_data[trial_data['state_marker'].isin([2, 3, 4])].copy()
    if df.empty:
        return None

    time_rel = df['time'] - trial_data['time'].iloc[0]
    max_t = time_rel.max()
    keep = time_rel <= (max_t - cutoff_s)
    df = df[keep].copy()
    time_rel = time_rel[keep]

    if df.empty:
        return None

    t_end = time_rel.max()

    # Glättung der Gaze
    df['gaze_x_smooth'] = df['gaze_x'].rolling(5, center=True, min_periods=1).mean()
    df['gaze_y_smooth'] = df['gaze_y'].rolling(5, center=True, min_periods=1).mean()

    # Einheitsvektor vom ersten Gaze-Punkt zum Target
    start_gaze_x = df['gaze_x_smooth'].iloc[0]
    start_gaze_y = df['gaze_y_smooth'].iloc[0]
    target_x = df['target_x'].iloc[0]
    target_y = df['target_y'].iloc[0]

    dx_target = target_x - start_gaze_x
    dy_target = target_y - start_gaze_y
    dist = np.sqrt(dx_target**2 + dy_target**2)
    if dist == 0:
        return None
    ux = dx_target / dist
    uy = dy_target / dist

    # Differenzen und radiale Geschwindigkeit
    df['dt_gaze'] = df['time'].diff()
    df['dx_gaze'] = df['gaze_x_smooth'].diff()
    df['dy_gaze'] = df['gaze_y_smooth'].diff()
    df['vx_gaze'] = df['dx_gaze'] / df['dt_gaze']
    df['vy_gaze'] = df['dy_gaze'] / df['dt_gaze']
    df['v_radial_gaze'] = df['vx_gaze'] * ux + df['vy_gaze'] * uy
    df.loc[df['v_radial_gaze'].abs() > 3500, 'v_radial_gaze'] = np.nan

    latency = peak = duration = quick_phase = slow_phase = pos_error = precision = np.nan

    mask = df['v_radial_gaze'].fillna(0) > threshold
    if mask.any():
        latency = time_rel[mask].iloc[0]
        t0 = latency
        t_peak = time_rel[df['v_radial_gaze'].idxmax()]
        duration = t_end - t0
        quick_phase = t_peak - t0
        slow_phase = t_end - t_peak

    peak = df['v_radial_gaze'].max()

    # Position error = letzter geglätteter Gaze-Punkt zum Target
    last = df.iloc[-1]
    pos_error = np.sqrt((last['gaze_x_smooth'] - target_x)**2 +
                        (last['gaze_y_smooth'] - target_y)**2)

    # Präzision = mittlerer Abstand aller Gaze-Punkte im State 4 zum Target
    df_state4 = df[df['state_marker'] == 4]
    if len(df_state4) > 0:
        dists = np.sqrt((df_state4['gaze_x_smooth'] - target_x)**2 +
                        (df_state4['gaze_y_smooth'] - target_y)**2)
        precision = dists.mean()

    return {
        'latency_gaze': latency,
        'peak_velocity_gaze': peak,
        'duration_gaze': duration,
        'quick_phase_gaze': quick_phase,
        'slow_phase_gaze': slow_phase,
        'position_error_gaze': pos_error,
        'precision_gaze': precision
    }

# rohdaten tabelle cursor metriken

still to do:

- gazue latency ist noch niedrig -> wahrscheinlich noch zeiten an cursor anpassen/ oder augen bewegung noch glätten/ bereinigen.

In [60]:
CUTOFF = 0.05   # Sekunden

# -----------------------------------------------------------------
# 1. Cursor-Metriken sammeln
# -----------------------------------------------------------------
cursor_metrics_list = []
for (p, tr, bl), grp in df_cc.groupby(['Participant', 'trial', 'block']):
    try:
        m = compute_cursor_metrics(grp, cutoff_s=CUTOFF)
        if m is not None:
            m['Participant'] = p
            m['trial'] = tr
            m['block'] = bl
            m['blob_width'] = grp['blob_width'].iloc[0]
            cursor_metrics_list.append(m)
    except Exception as e:
        print(f"Fehler Cursor {p}, B{bl}, T{tr}: {e}")

df_cursor_raw = pd.DataFrame(cursor_metrics_list)

# -----------------------------------------------------------------
# 2. Gaze-Metriken sammeln
# -----------------------------------------------------------------
gaze_metrics_list = []
for (p, tr, bl), grp in df_cc.groupby(['Participant', 'trial', 'block']):
    try:
        m = compute_gaze_metrics_for_trial(grp, cutoff_s=CUTOFF)
        if m is not None:
            m['Participant'] = p
            m['trial'] = tr
            m['block'] = bl
            m['blob_width'] = grp['blob_width'].iloc[0]
            gaze_metrics_list.append(m)
    except Exception as e:
        print(f"Fehler Gaze {p}, B{bl}, T{tr}: {e}")

df_gaze_raw = pd.DataFrame(gaze_metrics_list)

# -----------------------------------------------------------------
# 3. Zusammenführen (inner join über alle Identifier)
# -----------------------------------------------------------------
df_metrics_all = df_cursor_raw.merge(
    df_gaze_raw,
    on=['Participant', 'trial', 'block', 'blob_width'],
    how='inner'
)

print(f"Kombinierte Metriken: {len(df_metrics_all)} Trials")

# -----------------------------------------------------------------
# 4. Aggregation: Pro Blob-Width
# -----------------------------------------------------------------
metric_cols = [c for c in df_metrics_all.columns
               if c not in ['Participant', 'trial', 'block', 'blob_width']]

df_agg_blob = df_metrics_all.groupby('blob_width')[metric_cols].agg(['mean', 'std'])
df_agg_blob.columns = ['_'.join(col) for col in df_agg_blob.columns]
df_agg_blob = df_agg_blob.reset_index()

print("\n--- Aggregierte Metriken pro Blob-Width ---")
display(df_agg_blob.round(2))

# -----------------------------------------------------------------
# 5. Aggregation: Pro Participant & Blob-Width
# -----------------------------------------------------------------
df_agg_part = df_metrics_all.groupby(['Participant', 'blob_width'])[metric_cols].agg(['mean', 'std'])
df_agg_part.columns = ['_'.join(col) for col in df_agg_part.columns]
df_agg_part = df_agg_part.reset_index()

print("\n--- Metriken pro Participant & Blob-Width ---")
display(df_agg_part.round(2))

# -----------------------------------------------------------------
# 6. Optional als CSV speichern
# -----------------------------------------------------------------
output_dir = Path("metrics_output")
output_dir.mkdir(exist_ok=True)

df_agg_blob.to_csv(output_dir / "metrics_by_blobwidth_cutoff.csv", index=False)
df_agg_part.to_csv(output_dir / "metrics_by_participant_blobwidth_cutoff.csv", index=False)
df_metrics_all.to_csv(output_dir / "trial_metrics_raw_cutoff.csv", index=False)

print(f"\nCSV-Dateien gespeichert in {output_dir}/")

KeyboardInterrupt: 